# Build Curated Layer

In [1]:
%%sql
CREATE SCHEMA IF NOT EXISTS Curated COMMENT 'Business-ready KPIs, facts, dims'

StatementMeta(, 96fefab2-dfe9-45ce-82d7-8876abecbe98, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
# improve DirectLake for Power BI reporting
spark.conf.get('spark.sql.parquet.vorder.default')
spark.conf.set('spark.sql.parquet.vorder.default', 'true')
spark.conf.get('spark.sql.parquet.vorder.default')

StatementMeta(, 96fefab2-dfe9-45ce-82d7-8876abecbe98, 4, Finished, Available, Finished, False)

'true'

## Sale Facts

In [3]:
%%sql
CREATE OR REPLACE TABLE Curated.Sales_Fact
AS
SELECT 
    Date, PaymentMethod, Country, Store, WineId, UnitPrice, Quantity, Discount, round(Quantity * UnitPrice * (1 - Discount/100),2)  as TotalAmount
FROM 
    Enriched.sales

StatementMeta(, 96fefab2-dfe9-45ce-82d7-8876abecbe98, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Wine Dimension

In [4]:
%%sql
CREATE OR REPLACE TABLE Curated.Wine_Dimension
AS
SELECT 
    WineId, WineCode, WineName, Vintage, Type, Color, Classification, Category, Country, Area, Producer
FROM 
    Enriched.wines

StatementMeta(, 96fefab2-dfe9-45ce-82d7-8876abecbe98, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### Date Dimention

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

start_date = '2025-01-01'
end_date = '2026-06-30'

# Calculate the number of days for the period
num_days = (spark.sql(f"SELECT datediff('{end_date}', '{start_date}') as n").collect()[0]['n']) + 1

df_calendar = (
    spark.range(0, num_days)
    .withColumn("Date", expr(f"date_add('{start_date}', cast(id as int))"))
    .withColumn("DateCode", date_format(col("Date"), "yyyyMMdd"))
    .withColumn("MonthNumber", month(col("Date")))
    .withColumn("MonthNameCzech",
        when(month(col("Date")) == 1, "Leden")
        .when(month(col("Date")) == 2, "Únor")
        .when(month(col("Date")) == 3, "Březen")
        .when(month(col("Date")) == 4, "Duben")
        .when(month(col("Date")) == 5, "Květen")
        .when(month(col("Date")) == 6, "Červen")
        .when(month(col("Date")) == 7, "Červenec")
        .when(month(col("Date")) == 8, "Srpen")
        .when(month(col("Date")) == 9, "Září")
        .when(month(col("Date")) == 10, "Říjen")
        .when(month(col("Date")) == 11, "Listopad")
        .otherwise("Prosinec")
    )
    .withColumn("MonthNameEnglish", date_format(col("Date"), "MMMM"))
    .withColumn("DayNumber", dayofmonth(col("Date")))
    .withColumn("DayNameCzech",
        when(dayofweek(col("Date")) == 1, "Neděle")
        .when(dayofweek(col("Date")) == 2, "Pondělí")
        .when(dayofweek(col("Date")) == 3, "Úterý")
        .when(dayofweek(col("Date")) == 4, "Středa")
        .when(dayofweek(col("Date")) == 5, "Čtvrtek")
        .when(dayofweek(col("Date")) == 6, "Pátek")
        .otherwise("Sobota")
    )
    .withColumn("DayNameEnglish", date_format(col("Date"), "EEEE"))
    .withColumn("Quarter", concat(lit("Q"), quarter(col("Date"))))
    .withColumn("YearQuarter", concat(year(col("Date")), lit("-Q"), quarter(col("Date"))))
    .withColumn("YearMonth", date_format(col("Date"), "yyyy-MM"))
    .withColumn("Year", date_format(col("Date"), "yyyy"))
    .drop("id")
    .orderBy("Date")
)


df_calendar.write.format("delta").mode("overwrite").saveAsTable("Curated.Date_Dimension")

display(df_calendar.limit(10))
print(f"Written {df_calendar.count()} rows.")